# 01 — Submission Review

Use this after `make predict`. It validates `submissions/submission.csv`, summarizes confidence/box counts, and overlays predictions on test images.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config" / "competition.yaml").is_file():
            return path
    raise FileNotFoundError("Could not find repo root containing config/competition.yaml")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(REPO_ROOT)

In [ ]:
import pandas as pd
from ua_detrac_starter.config import config_path, load_config

cfg, _ = load_config("config/competition.yaml")
sample_path = config_path(cfg, "sample_submission", "data/competition_starter/sample_submission.csv")
submission_path = config_path(cfg, "submission_csv", "submissions/submission.csv")
test_images = config_path(cfg, "test_images", "data/competition_starter/data/test/images")

print("sample:", sample_path, sample_path.is_file())
print("submission:", submission_path, submission_path.is_file())
print("test images:", test_images, test_images.is_dir())

In [ ]:
if sample_path.is_file() and submission_path.is_file():
    result = subprocess.run(
        ["python3", "scripts/validate_submission.py", "--sample", str(sample_path), "--submission", str(submission_path)],
        cwd=REPO_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
else:
    print("Run make predict first, then rerun this cell.")

In [ ]:
if submission_path.is_file():
    submission = pd.read_csv(submission_path, dtype=str, keep_default_na=False, na_filter=False)
else:
    submission = pd.DataFrame(columns=["id", "image_id", "prediction_string"])

def parse_prediction_string(row):
    value = row["prediction_string"]
    if value == "no box":
        return []
    parts = value.split()
    if len(parts) % 6 != 0:
        return [{"id": row["id"], "image_id": row["image_id"], "parse_error": f"{len(parts)} tokens"}]
    parsed = []
    for start in range(0, len(parts), 6):
        class_id, conf, x_center, y_center, width, height = parts[start:start + 6]
        parsed.append({
            "id": row["id"],
            "image_id": row["image_id"],
            "class_id": int(class_id),
            "confidence": float(conf),
            "x_center": float(x_center),
            "y_center": float(y_center),
            "width": float(width),
            "height": float(height),
            "area": float(width) * float(height),
        })
    return parsed

prediction_rows = []
for _, row in submission.iterrows():
    prediction_rows.extend(parse_prediction_string(row))
predictions = pd.DataFrame(prediction_rows)

print(f"rows: {len(submission):,}")
print(f"detections: {len(predictions):,}")
print(f"no-box rows: {(submission['prediction_string'] == 'no box').sum() if not submission.empty else 0:,}")
predictions.head()

In [ ]:
if predictions.empty:
    print("No predictions to summarize.")
else:
    display(predictions.groupby("class_id").agg(
        detections=("class_id", "size"),
        mean_conf=("confidence", "mean"),
        median_area=("area", "median"),
    ).round(4))
    display(predictions.groupby("image_id").size().describe().to_frame("detections_per_image").round(2))

In [ ]:
import matplotlib.pyplot as plt

if not predictions.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    predictions["confidence"].plot(kind="hist", bins=40, ax=axes[0], title="Confidence distribution")
    predictions["area"].clip(upper=0.2).plot(kind="hist", bins=40, ax=axes[1], title="Box area distribution")
    axes[0].set_xlabel("confidence")
    axes[1].set_xlabel("normalized area")
    plt.tight_layout()

In [ ]:
from PIL import Image
from matplotlib.patches import Rectangle

IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
CLASS_NAMES = {0: "truck", 1: "car", 2: "van", 3: "bus"}

def find_image(image_id: str) -> Path | None:
    raw = Path(image_id)
    if raw.suffix:
        candidate = test_images / raw.name
        return candidate if candidate.is_file() else None
    for extension in IMAGE_EXTS:
        candidate = test_images / f"{image_id}{extension}"
        if candidate.is_file():
            return candidate
    return None

def draw_predictions(image_id: str, ax):
    image_path = find_image(image_id)
    if image_path is None:
        ax.set_title(f"missing: {image_id}")
        ax.axis("off")
        return
    image = Image.open(image_path).convert("RGB")
    image_width, image_height = image.size
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis("off")
    subset = predictions[predictions["image_id"] == image_id].sort_values("confidence", ascending=False)
    for _, pred in subset.iterrows():
        left = (pred.x_center - pred.width / 2) * image_width
        top = (pred.y_center - pred.height / 2) * image_height
        rect = Rectangle((left, top), pred.width * image_width, pred.height * image_height, fill=False, edgecolor="yellow", linewidth=2)
        ax.add_patch(rect)
        label = f"{CLASS_NAMES.get(int(pred.class_id), int(pred.class_id))} {pred.confidence:.2f}"
        ax.text(left, max(0, top - 3), label, color="black", backgroundcolor="yellow", fontsize=8)

if predictions.empty:
    print("No predictions to draw.")
else:
    top_images = predictions.sort_values("confidence", ascending=False)["image_id"].drop_duplicates().head(6).tolist()
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, image_id in zip(axes.flat, top_images):
        draw_predictions(image_id, ax)
    for ax in axes.flat[len(top_images):]:
        ax.axis("off")
    plt.tight_layout()